<cell_type>markdown</cell_type># Feature Engineering - PCA y Feature Selection

**Proyecto:** Sistema de Clasificación de Acciones S&P 500

**Grupo 27** - Universidad de Los Andes

---

## Objetivo

Evaluar si reducción de dimensionalidad mejora el desempeño de los 3 modelos:
1. PCA con 5, 10, 15 componentes
2. Feature Selection (Top 5, 10, 15 features por importancia)
3. Comparar con modelo completo (17 features)

**Modelos a evaluar:** Logistic Regression, Random Forest, XGBoost (los 3 en paralelo)

In [1]:
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
)

import warnings
warnings.filterwarnings('ignore')

## 1. Configuración

In [2]:
# Configurar MLflow
experiment_name = "/sp500-feature-engineering"
mlflow.set_experiment(experiment_name)

# Configuración de modelos (solo random_state)
models_config = {
    'Logistic Regression': (LogisticRegression, {'random_state': 42}),
    'Random Forest': (RandomForestClassifier, {'random_state': 42}),
    'XGBoost': (XGBClassifier, {'random_state': 42})
}

print(f"Experimento: {experiment_name}")
print(f"Modelos a evaluar: {list(models_config.keys())}")

Experimento: /sp500-feature-engineering
Modelos a evaluar: ['Logistic Regression', 'Random Forest', 'XGBoost']


## 2. Carga de Datos

In [ ]:
# Cargar datasets
train = pd.read_parquet('./data/processed/ml_ready/train.parquet')
test = pd.read_parquet('./data/processed/ml_ready/test.parquet')

# Separar features y target
feature_cols = [col for col in train.columns if col not in ['Ticker', 'Date', 'Target']]

X_train = train[feature_cols]
y_train = train['Target']
X_test = test[feature_cols]
y_test = test['Target']

print(f"Features originales: {len(feature_cols)}")

Features originales: 22


In [4]:
# Función helper para entrenar y evaluar
def train_and_evaluate(model_class, params, X_train, y_train, X_test, y_test):
    """
    Entrena modelo y retorna métricas.
    """
    model = model_class(**params)
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    
    metrics = {
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred),
        'recall': recall_score(y_test, y_pred),
        'f1_score': f1_score(y_test, y_pred),
        'roc_auc': roc_auc_score(y_test, y_pred_proba)
    }
    
    return metrics, model

## 3. Experimentos con PCA

Evaluar PCA con 5, 10 y 15 componentes principales.

In [5]:
# Normalizar datos (requerido para PCA)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Datos normalizados para PCA")

Datos normalizados para PCA


In [6]:
# Almacenar resultados
resultados = []

# Probar PCA con diferentes números de componentes
n_components_list = [5, 10, 15]

for modelo_nombre, (modelo_class, params) in models_config.items():
    print(f"\n{modelo_nombre}")
    
    for n_comp in n_components_list:
        
        with mlflow.start_run(run_name=f"{modelo_nombre}_PCA_{n_comp}"):
            
            # Aplicar PCA
            pca = PCA(n_components=n_comp, random_state=42)
            X_train_pca = pca.fit_transform(X_train_scaled)
            X_test_pca = pca.transform(X_test_scaled)
            
            # Varianza explicada
            varianza = pca.explained_variance_ratio_.sum()
            
            # Entrenar y evaluar
            metrics, model = train_and_evaluate(
                modelo_class, params, 
                X_train_pca, y_train, 
                X_test_pca, y_test
            )
            
            # Registrar en MLflow
            mlflow.log_param("modelo", modelo_nombre)
            mlflow.log_param("config", f"PCA-{n_comp}")
            mlflow.log_param("n_components", n_comp)
            mlflow.log_param("varianza_explicada", varianza)
            for metric_name, metric_value in metrics.items():
                mlflow.log_metric(metric_name, metric_value)
            
            # Guardar resultados
            resultados.append({
                'Modelo': modelo_nombre,
                'Config': f'PCA-{n_comp}',
                'Features': n_comp,
                'Varianza': f"{varianza:.2%}",
                'Accuracy': metrics['accuracy'],
                'Precision': metrics['precision'],
                'Recall': metrics['recall'],
                'F1-Score': metrics['f1_score'],
                'ROC-AUC': metrics['roc_auc']
            })
            
            print(f"  PCA-{n_comp}: ROC-AUC={metrics['roc_auc']:.4f}, Var={varianza:.2%}")


Logistic Regression
  PCA-5: ROC-AUC=0.5119, Var=86.07%
  PCA-10: ROC-AUC=0.5052, Var=99.34%
  PCA-15: ROC-AUC=0.5044, Var=100.00%

Random Forest
  PCA-5: ROC-AUC=0.5109, Var=86.07%
  PCA-10: ROC-AUC=0.5021, Var=99.34%
  PCA-15: ROC-AUC=0.4979, Var=100.00%

XGBoost
  PCA-5: ROC-AUC=0.5078, Var=86.07%
  PCA-10: ROC-AUC=0.4993, Var=99.34%
  PCA-15: ROC-AUC=0.4999, Var=100.00%


In [7]:
# Obtener feature importance usando Random Forest
rf_temp = RandomForestClassifier(random_state=42)
rf_temp.fit(X_train, y_train)

feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': rf_temp.feature_importances_
}).sort_values('importance', ascending=False)

print("Top 10 features por importancia:")
print(feature_importance.head(10).to_string(index=False))

Top 10 features por importancia:
      feature  importance
Volume_change    0.062666
      Returns    0.062073
       RSI_14    0.057100
     BB_width    0.056649
    MACD_diff    0.056632
Volatility_10    0.056406
       Volume    0.056032
          OBV    0.054338
  MACD_signal    0.051023
         MACD    0.050927


## 4. Experimentos con Feature Selection

Feature Selection basado en importancia de Random Forest.

In [8]:
# Probar Feature Selection con diferentes números de features
n_features_list = [5, 10, 15]

for modelo_nombre, (modelo_class, params) in models_config.items():
    print(f"\n{modelo_nombre}")
    
    for n_feat in n_features_list:
        
        with mlflow.start_run(run_name=f"{modelo_nombre}_Top_{n_feat}"):
            
            # Seleccionar top N features
            top_features = feature_importance.head(n_feat)['feature'].tolist()
            
            X_train_fs = X_train[top_features]
            X_test_fs = X_test[top_features]
            
            # Entrenar y evaluar
            metrics, model = train_and_evaluate(
                modelo_class, params,
                X_train_fs, y_train,
                X_test_fs, y_test
            )
            
            # Registrar en MLflow
            mlflow.log_param("modelo", modelo_nombre)
            mlflow.log_param("config", f"Top-{n_feat}")
            mlflow.log_param("n_features", n_feat)
            mlflow.log_param("selected_features", top_features)
            for metric_name, metric_value in metrics.items():
                mlflow.log_metric(metric_name, metric_value)
            
            # Guardar resultados
            resultados.append({
                'Modelo': modelo_nombre,
                'Config': f'Top-{n_feat}',
                'Features': n_feat,
                'Varianza': 'N/A',
                'Accuracy': metrics['accuracy'],
                'Precision': metrics['precision'],
                'Recall': metrics['recall'],
                'F1-Score': metrics['f1_score'],
                'ROC-AUC': metrics['roc_auc']
            })
            
            print(f"  Top-{n_feat}: ROC-AUC={metrics['roc_auc']:.4f}")


Logistic Regression
  Top-5: ROC-AUC=0.5112
  Top-10: ROC-AUC=0.4915
  Top-15: ROC-AUC=0.4915

Random Forest
  Top-5: ROC-AUC=0.4895
  Top-10: ROC-AUC=0.4940
  Top-15: ROC-AUC=0.4958

XGBoost
  Top-5: ROC-AUC=0.5010
  Top-10: ROC-AUC=0.5008
  Top-15: ROC-AUC=0.5003


## 5. Comparación de Resultados

Top 10 mejores configuraciones basados en ROC-AUC

In [9]:
# Consolidar todos los resultados
df_comparacion = pd.DataFrame(resultados)

# Ordenar por ROC-AUC (métrica más robusta)
df_comparacion = df_comparacion.sort_values('ROC-AUC', ascending=False).reset_index(drop=True)

print("\nComparación de TOP 10 Configuraciones por ROC-AUC:")
print(df_comparacion.head(10).to_string(index=False))


Comparación de TOP 10 Configuraciones por ROC-AUC:
             Modelo Config  Features Varianza  Accuracy  Precision   Recall  F1-Score  ROC-AUC
Logistic Regression  PCA-5         5   86.07%  0.518084   0.518102 0.958061  0.672519 0.511854
Logistic Regression  Top-5         5      N/A  0.517289   0.516932 0.998461  0.681192 0.511163
      Random Forest  PCA-5         5   86.07%  0.510135   0.522348 0.602539  0.559585 0.510880
            XGBoost  PCA-5         5   86.07%  0.504968   0.517363 0.619084  0.563671 0.507842
Logistic Regression PCA-10        10   99.34%  0.520469   0.518780 0.988457  0.680440 0.505211
Logistic Regression PCA-15        15  100.00%  0.519475   0.518228 0.989996  0.680328 0.504397
      Random Forest PCA-10        10   99.34%  0.498410   0.512108 0.610235  0.556882 0.502124
            XGBoost  Top-5         5      N/A  0.509340   0.520531 0.634090  0.571726 0.500965
            XGBoost Top-10        10      N/A  0.497019   0.513199 0.508657  0.510918 0.50080

## 6. Resultados finales

In [10]:
# Mejor configuración global
mejor_fila = df_comparacion.iloc[0]
print(f"\nMejor configuración global (por ROC-AUC):")
print(f"  Modelo: {mejor_fila['Modelo']}")
print(f"  Config: {mejor_fila['Config']}")
print(f"  ROC-AUC: {mejor_fila['ROC-AUC']:.4f}")
print(f"  F1-Score: {mejor_fila['F1-Score']:.4f}")

# Mejor por modelo
print("\n\nMejor configuración por modelo (por ROC-AUC):")
for modelo in ['Logistic Regression', 'Random Forest', 'XGBoost']:
    df_modelo = df_comparacion[df_comparacion['Modelo'] == modelo]
    mejor_modelo = df_modelo.iloc[0]
    print(f"  {modelo}: {mejor_modelo['Config']} (ROC-AUC={mejor_modelo['ROC-AUC']:.4f})")


Mejor configuración global (por ROC-AUC):
  Modelo: Logistic Regression
  Config: PCA-5
  ROC-AUC: 0.5119
  F1-Score: 0.6725


Mejor configuración por modelo (por ROC-AUC):
  Logistic Regression: PCA-5 (ROC-AUC=0.5119)
  Random Forest: PCA-5 (ROC-AUC=0.5109)
  XGBoost: PCA-5 (ROC-AUC=0.5078)


## 7. Análisis y Decisión

### Decisión para Notebook 03:

Basándose en los resultados de ROC-AUC, se seleccionarán las mejores combinaciones modelo + configuración de features para optimizar con Optuna.

## Conclusiones

- Se evaluaron 21 configuraciones (3 modelos × 7 configs)
- Todos los experimentos registrados en MLflow
- Resultados ordenados por ROC-AUC (métrica más confiable que F1-Score)
